In [1]:
# put this at the top of every notebook
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics as qa
from yfinance_api3.modules.montecarlo import simulate, compare_methods
import yfinance_api3.modules.plots as plots

In [3]:
client = StockClient(ttl_price=60, ttl_info=900)
quant = qa(client)
symbols = ["AAPL", "MSFT", "NVDA", "GOOGL", "JPM"]
weights = [0.25, 0.25, 0.20, 0.15, 0.15]

In [4]:
# single run
result = simulate(quant, symbols, weights,
    horizon=252,        # 1 year forward
    n_sims=2000,
    method="historical",
    period="3y")

print(result)
plots.monte_carlo(result).show()

MonteCarloResult — historical  (2,000 sims, 252d horizon)
  Median final value : $140,511  (+40.5%)
  Mean final value   : $142,545
  5th pct (bad)      : $98,930  (-1.1%)
  95th pct (good)    : $190,510  (+90.5%)
  VaR 95% (horizon)  : -1.07%
  CVaR 95% (horizon) : -8.97%
  Prob of gain       : 94.2%
  Prob of loss > 10% : 1.6%
  Prob of loss > 20% : 0.5%


In [5]:
# compare all three distributional assumptions in one table
df = compare_methods(quant, symbols, weights, horizon=252, n_sims=2000)
print(df[["var_95", "cvar_95", "prob_gain", "prob_loss_10pct"]])

              var_95   cvar_95  prob_gain  prob_loss_10pct
historical -0.010704 -0.089695     0.9415           0.0160
normal      0.003192 -0.082471     0.9500           0.0155
t_dist     -0.137894 -0.214719     0.8790           0.0675


In [6]:
quant.monte_carlo('INTC',horizon_years=2,n_sims=10000,target_gain=0.5,max_drawdown_limit=.30,history_period="5y",drift_override=.15,volatility_override=.22)

{'symbol': 'INTC',
 'entry_price': 82.54000091552734,
 'horizon_years': 2,
 'n_sims': 10000,
 'mu_annual': 0.15,
 'sigma_annual': 0.22,
 'target_gain': 0.5,
 'max_drawdown_limit': 0.3,
 'prob_gain': 0.316,
 'prob_drawdown_ok': 0.7777,
 'prob_both': 0.3098,
 'prob_loss': 0.2085,
 'median_return': 0.2933298266134439,
 'percentile_10': -0.13674896446717588,
 'percentile_90': 0.931696327033752,
 'expected_return': 0.35557685973258196}